# Compression & Loss: What Survives Reduction?

Every analysis we've done so far is a **compression**. We take a large, messy text and reduce it to something smaller:

- Word counts
- Motif profiles
- Agency summaries
- Structured schemas

Compression is useful—it lets us compare, aggregate, and reason about texts at scale.

But compression is also **lossy**. Something always gets left behind.

This notebook asks: **What survives? What vanishes? What gets invented?**

---

## The Central Tension

- **Close reading** preserves nuance but doesn't scale
- **Distant reading** scales but loses nuance

Neither is wrong. The question is: what are you willing to lose for what you gain?

---

## 1. Setup

In [ ]:
from collections import Counter
import re

# A short poem for close examination
poem = """The Tyger
by William Blake

Tyger Tyger, burning bright,
In the forests of the night;
What immortal hand or eye,
Could frame thy fearful symmetry?

In what distant deeps or skies,
Burnt the fire of thine eyes?
On what wings dare he aspire?
What the hand, dare seize the fire?

And what shoulder, & what art,
Could twist the sinews of thy heart?
And when thy heart began to beat,
What dread hand? & what dread feet?

What the hammer? what the chain,
In what furnace was thy brain?
What the anvil? what dread grasp,
Dare its deadly terrors clasp!

When the stars threw down their spears
And water'd heaven with their tears:
Did he smile his work to see?
Did he who made the Lamb make thee?

Tyger Tyger burning bright,
In the forests of the night:
What immortal hand or eye,
Dare frame thy fearful symmetry?"""

print(poem)

---

## 2. Compression Level 1: Word Frequency

The simplest compression: which words appear most often?

In [ ]:
def get_words(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

words = get_words(poem)
word_counts = Counter(words)

print("Top 15 words:")
for word, count in word_counts.most_common(15):
    print(f"  {word}: {count}")

### What survives?

- The word "what" dominates—the poem is structured around questions
- "tyger" appears (the subject)
- "thy" and "the" (function words)

### What's lost?

- The **order** of words
- The **sound** ("burning bright" alliterates)
- The **questions** themselves (we see "what" but not what's being asked)
- The **rhythm** and **rhyme**
- The **imagery** (fire, forests, night)

---

## 3. Compression Level 2: Motif Buckets

Let's group words into interpretive categories.

In [ ]:
MOTIFS = {
    "fire": ["fire", "burning", "burnt", "bright", "furnace", "hammer", "anvil"],
    "body": ["hand", "eye", "eyes", "shoulder", "heart", "feet", "sinews", "brain"],
    "creation": ["frame", "made", "twist", "seize", "clasp", "work"],
    "fear": ["fearful", "dread", "deadly", "terrors", "dare"],
    "cosmic": ["immortal", "heaven", "stars", "night", "skies", "deeps"],
    "animal": ["tyger", "lamb", "wings"]
}

def count_motifs(words, motifs):
    results = {}
    for name, bucket in motifs.items():
        count = sum(1 for w in words if w in bucket)
        results[name] = count
    return results

motif_counts = count_motifs(words, MOTIFS)

print("Motif profile:")
for motif, count in sorted(motif_counts.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * count
    print(f"  {motif:12} {bar} {count}")

### What survives?

- The poem's concern with **body parts** (the creator's hands, eyes)
- The **fire/forge** imagery
- The **fear/dread** vocabulary

### What's lost?

- The **relationship** between motifs (fire + body = a blacksmith god)
- The **questions** (is this creator good or terrifying?)
- The **Lamb** reference (to Blake's companion poem)
- The **ambiguity** (the poem doesn't answer its own questions)

---

## 4. Compression Level 3: A Structured Summary

Let's compress the poem into a structured profile like we did for the poetry pipeline.

In [ ]:
# A human-created structured summary
tyger_profile = {
    "title": "The Tyger",
    "author": "William Blake",
    "line_count": 24,
    "stanza_count": 6,
    "tone": "awe",
    "secondary_tone": "fear",
    "devices": ["repetition", "anaphora", "alliteration", "imagery", "apostrophe"],
    "speaker_stance": "questioner",
    "dominant_sense": "sight",
    "imagery": ["tiger", "fire", "forge", "night", "stars", "lamb"],
    "themes": ["creation", "good and evil", "divine power"],
    "has_volta": True,  # The Lamb stanza is a turn
    "keywords": ["tyger", "burning", "fearful", "symmetry", "dare"]
}

import json
print(json.dumps(tyger_profile, indent=2))

### What survives?

- Key **metadata** (title, author, length)
- **Tone** labels (awe, fear)
- **Device** identification
- **Theme** tags

### What's lost?

- The **actual poem** (you can't reconstruct it from this)
- The **specific questions** Blake asks
- The **sound** of "Tyger Tyger, burning bright"
- The **theological complexity** (is God responsible for evil?)
- The **intertextuality** (relationship to "The Lamb")
- **Your experience** of reading it

---

## 5. Compression Level 4: A Single Sentence

The most extreme compression: summarize the poem in one sentence.

In [ ]:
# Different one-sentence summaries
summaries = [
    "A speaker asks who could have created the terrifying tiger.",
    "Blake questions whether the same God who made the gentle Lamb also made the fearsome Tyger.",
    "The poem explores the problem of evil through the image of a burning tiger.",
    "A series of unanswered questions about divine creation and its moral implications.",
    "Tyger = scary, God = mysterious, questions = many."
]

print("Five one-sentence summaries:\n")
for i, s in enumerate(summaries, 1):
    print(f"{i}. {s}\n")

### Which summary is "correct"?

They all capture *something*. They all lose *almost everything*.

The choice of summary reflects the summarizer's priorities:
- Summary 1: focuses on the speaker and subject
- Summary 2: focuses on the Lamb connection (intertextuality)
- Summary 3: focuses on theological theme
- Summary 4: focuses on form (questions) and ambiguity
- Summary 5: is reductive to the point of comedy

**Compression is interpretation.**

---

## 6. The Reconstruction Test

Here's a way to measure loss: **can you reconstruct the original from the compression?**

In [ ]:
# What can you reconstruct from word frequencies alone?
print("From word frequencies, you might guess:")
print("  - The poem asks many questions ('what' is most common)")
print("  - It's about a tiger ('tyger' appears)")
print("  - It uses archaic language ('thy', 'thine')")
print()
print("You could NOT reconstruct:")
print("  - The actual lines")
print("  - The rhyme scheme")
print("  - The specific questions asked")
print("  - The emotional arc")

In [ ]:
# What can you reconstruct from the structured profile?
print("From the structured profile, you might write:")
print()

reconstruction_prompt = f"""
Write a poem with:
- Title: {tyger_profile['title']}
- Tone: {tyger_profile['tone']} with {tyger_profile['secondary_tone']}
- Devices: {', '.join(tyger_profile['devices'])}
- Imagery: {', '.join(tyger_profile['imagery'])}
- Themes: {', '.join(tyger_profile['themes'])}
- Length: {tyger_profile['line_count']} lines
"""

print(reconstruction_prompt)

If you gave this prompt to an AI (or a human), they could write *a* poem about a tiger with fire imagery and themes of creation.

But it wouldn't be *this* poem. The specific genius of Blake's language, rhythm, and theological provocation would be lost.

**The profile is a recipe, not the dish.**

---

## 7. What Gets Invented?

Compression doesn't just lose information—it sometimes **adds** information that wasn't there.

When we label the tone as "awe," we're making an interpretive claim. Someone else might say "terror" or "wonder" or "irony."

The label is an **invention**—a category we impose on the text.

In [ ]:
# Different interpreters might produce different profiles
alternative_profile = {
    "title": "The Tyger",
    "author": "William Blake",
    "tone": "irony",  # Different!
    "secondary_tone": "wonder",  # Different!
    "themes": ["industrialization", "romanticism", "critique of rationalism"],  # Different!
    "speaker_stance": "skeptic",  # Different!
}

print("Original profile tone:", tyger_profile["tone"])
print("Alternative profile tone:", alternative_profile["tone"])
print()
print("Original themes:", tyger_profile["themes"])
print("Alternative themes:", alternative_profile["themes"])

Both profiles are **defensible**. Neither is **neutral**.

The structured output doesn't just compress the poem—it **interprets** it. And interpretation always involves invention.

---

## 8. The Trade-Off Table

Let's be explicit about what each compression level trades.

In [ ]:
tradeoffs = [
    {
        "level": "Full text",
        "size": "~1000 chars",
        "preserves": "Everything",
        "loses": "Nothing",
        "enables": "Close reading",
        "prevents": "Comparison at scale"
    },
    {
        "level": "Word frequencies",
        "size": "~50 pairs",
        "preserves": "Vocabulary, emphasis",
        "loses": "Order, syntax, sound",
        "enables": "Vocabulary comparison",
        "prevents": "Understanding meaning"
    },
    {
        "level": "Motif counts",
        "size": "~10 numbers",
        "preserves": "Thematic emphasis",
        "loses": "Specific words, relationships",
        "enables": "Thematic comparison",
        "prevents": "Nuanced interpretation"
    },
    {
        "level": "Structured profile",
        "size": "~20 fields",
        "preserves": "Labeled features",
        "loses": "The text itself",
        "enables": "Aggregation, generation",
        "prevents": "Verification without original"
    },
    {
        "level": "One sentence",
        "size": "~100 chars",
        "preserves": "A claim about the text",
        "loses": "Almost everything",
        "enables": "Quick reference",
        "prevents": "Any real understanding"
    }
]

print("Compression Trade-Offs\n")
for t in tradeoffs:
    print(f"--- {t['level']} ({t['size']}) ---")
    print(f"  Preserves: {t['preserves']}")
    print(f"  Loses: {t['loses']}")
    print(f"  Enables: {t['enables']}")
    print(f"  Prevents: {t['prevents']}")
    print()

---

## 9. When Is Compression Worth It?

Compression is worth it when:

1. **You need to compare many texts** (can't close-read 1000 poems)
2. **You want to find patterns** (individual readings miss corpus-level trends)
3. **You're building something** (generation needs structured input)
4. **You're teaching** (simplified models help beginners)

Compression is NOT worth it when:

1. **You need the specific language** (legal, liturgical, poetic contexts)
2. **Ambiguity matters** (the poem's power is in what it doesn't resolve)
3. **You're making high-stakes claims** (scholarship requires evidence)
4. **The compression hides your interpretive choices** (bad faith)

---

## ✏️ Exercise: Compress and Reflect

Take a text you care about. Compress it at multiple levels. Then ask:

- What did each level preserve?
- What did each level lose?
- What did each level invent?
- Which level is most useful for your purpose?

In [ ]:
# Your text
my_text = """
Paste a short text here (a poem, a paragraph, a passage).
"""

# Level 1: Word frequencies
my_words = get_words(my_text)
my_freqs = Counter(my_words).most_common(10)
print("Level 1 - Top 10 words:")
print(my_freqs)

# Level 2: One-sentence summary (you write this)
my_summary = "Write your one-sentence summary here."
print(f"\nLevel 2 - Summary: {my_summary}")

# Level 3: Structured profile (you fill this in)
my_profile = {
    "tone": "",
    "themes": [],
    "keywords": []
}
print(f"\nLevel 3 - Profile: {my_profile}")

---

## Summary: Compression Is a Choice

| Principle | Implication |
|-----------|-------------|
| Compression is lossy | You always lose something |
| Compression is interpretive | You always add something |
| Compression enables scale | You can compare what you couldn't before |
| Compression hides choices | Your categories shape what you see |

**The goal is not to avoid compression. The goal is to know what you're trading.**

---

## Reflection Questions

- What would a "lossless" compression of a poem look like? Is it possible?
- When you summarize a text in conversation, what do you preserve?
- How do different disciplines compress texts differently? (Literary criticism vs. linguistics vs. history)
- If an AI compresses a text, whose interpretation is it?

---

## Next: Schemas and Validation

Now that you've felt the tension between compression and loss, we'll formalize structure with **schemas**—explicit templates that define what a compression should look like.

→ Continue to `05_schemas_and_validation.ipynb`